# Phase 5 — Complaint & Topic Analysis

Keyword-based complaint classification, TF-IDF keyword extraction, and **LDA topic modeling** on Phase 2 / Phase 4 cleaned text.

**Inputs (auto-detected):**
- `sentiment_results_phase4.csv` (Phase 4) — preferred if present
- `cleaned_reviews_phase2.csv` (Phase 2) — fallback

**Output:** `complaint_topic_results_phase5.csv`

```bash
pip install pandas scikit-learn nltk
python -c "import nltk; nltk.download('stopwords'); nltk.download('punkt')"
```

## Install dependencies (run once)

In [2]:
%pip install -q pandas scikit-learn nltk
import nltk
for pkg in ("stopwords", "punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
print("Using sklearn LDA fallback (no gensim).")


Note: you may need to restart the kernel to use updated packages.
Using sklearn LDA fallback (no gensim).


## Setup — imports, configuration, and helpers

In [1]:
from __future__ import annotations

import gc
import re
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd().resolve()
PHASE2_CSV = PROJECT_DIR / "cleaned_reviews_phase2.csv"
PHASE4_CSV = PROJECT_DIR / "sentiment_results_phase4.csv"
OUTPUT_CSV = PROJECT_DIR / "complaint_topic_results_phase5.csv"

RANDOM_STATE = 42
TOP_N_TFIDF = 20
TOP_N_PER_REVIEW = 5
LDA_NUM_TOPICS = 8
LDA_PASSES = 10

# None = all rows for rule-based steps; LDA uses LDA_MAX_DOCS for training speed
MAX_ROWS: Optional[int] = None
LDA_MAX_DOCS: Optional[int] = 100_000
OUTPUT_MAX_ROWS: Optional[int] = None

SENTIMENT_MAP = {0: "negative", 2: "neutral", 4: "positive", "Negative": "negative", "Neutral": "neutral", "Positive": "positive"}

print(f"Project directory: {PROJECT_DIR}")


Project directory: D:\NCPL\Bootcamp\Projects\Project 5\new data\data 2


---
## Step 1 — Load cleaned data

Load Phase 4 results if available; otherwise Phase 2. Standardize `cleaned_review_text` and optional `sentiment`, then drop missing text.

In [3]:
def safe_text(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_sentiment(value) -> str:
    if pd.isna(value):
        return "unknown"
    if isinstance(value, str) and value.strip():
        key = value.strip()
        return SENTIMENT_MAP.get(key, key.lower())
    try:
        return SENTIMENT_MAP.get(int(value), "unknown")
    except (TypeError, ValueError):
        return "unknown"


def load_analysis_data() -> pd.DataFrame:
    if PHASE4_CSV.is_file():
        print(f"Loading Phase 4 output: {PHASE4_CSV.name}")
        header = pd.read_csv(PHASE4_CSV, nrows=0)
        cols = set(header.columns)
        usecols = [c for c in ("cleaned_review_text", "predicted_sentiment", "target", "row_id") if c in cols]
        if "cleaned_review_text" not in usecols:
            raise ValueError(f"{PHASE4_CSV.name} missing cleaned_review_text column.")
        df = pd.read_csv(PHASE4_CSV, usecols=usecols)
        df["cleaned_review_text"] = df["cleaned_review_text"].map(safe_text)
        if "predicted_sentiment" in df.columns:
            df["sentiment"] = df["predicted_sentiment"].map(normalize_sentiment)
        elif "target" in df.columns:
            df["sentiment"] = df["target"].map(normalize_sentiment)
        else:
            df["sentiment"] = "unknown"
    elif PHASE2_CSV.is_file():
        print(f"Loading Phase 2 output: {PHASE2_CSV.name}")
        header = pd.read_csv(PHASE2_CSV, nrows=0)
        text_col = next(
            (c for c in ("text_lemmatized", "text_clean", "text", "review_text") if c in header.columns),
            None,
        )
        if text_col is None:
            raise ValueError(f"No text column found in {PHASE2_CSV.name}")
        usecols = [text_col]
        if "target" in header.columns:
            usecols.append("target")
        if "row_id" in header.columns:
            usecols.insert(0, "row_id")
        df = pd.read_csv(PHASE2_CSV, usecols=usecols)
        df["cleaned_review_text"] = df[text_col].map(safe_text)
        if text_col != "cleaned_review_text":
            df.drop(columns=[text_col], inplace=True, errors="ignore")
        if "target" in df.columns:
            df["sentiment"] = df["target"].map(normalize_sentiment)
        else:
            df["sentiment"] = "unknown"
    else:
        raise FileNotFoundError(
            "No input found. Run phase2_text_cleaning.ipynb or phase4_sentiment_analysis.ipynb first."
        )

    missing = df["cleaned_review_text"].eq("") | df["cleaned_review_text"].str.lower().eq("missing_review")
    if missing.any():
        print(f"Dropping {missing.sum():,} rows with missing text.")
        df = df.loc[~missing]

    if MAX_ROWS is not None and len(df) > MAX_ROWS:
        df = df.sample(MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f"Subsampled to {MAX_ROWS:,} rows (MAX_ROWS).")

    df.reset_index(drop=True, inplace=True)
    gc.collect()
    return df


try:
    df = load_analysis_data()
    print(f"Loaded {len(df):,} rows | columns: {list(df.columns)}")
    display(df[["cleaned_review_text", "sentiment"]].head(3))
except (FileNotFoundError, ValueError) as exc:
    print(f"Load error: {exc}", file=sys.stderr)
    raise


Loading Phase 2 output: cleaned_reviews_phase2.csv
Dropping 352 rows with missing text.
Loaded 1,565,497 rows | columns: ['row_id', 'target', 'cleaned_review_text', 'sentiment']


,cleaned_review_text,sentiment
0,switchfoot awww bummer shoulda get david carr ...,negative
1,upset update facebook texting might cry result...,negative
2,kenichan dive many time ball manage save 50 re...,negative


---
## Section A — Keyword-based complaint classification

### Step 2 — Define complaint categories

Rule-based keyword dictionaries map review text to complaint themes. A review may match **multiple** categories.

In [4]:
COMPLAINT_CATEGORIES: Dict[str, List[str]] = {
    "delivery_delay": ["late", "delay", "slow delivery", "not delivered", "shipping issue", "shipping", "deliver"],
    "poor_quality": ["bad quality", "broken", "damaged", "defective", "poor build", "poor", "cheap", "fake"],
    "high_price": ["expensive", "overpriced", "high price", "costly", "price", "fee"],
    "customer_service_issue": ["rude", "support", "customer service", "unhelpful", "service", "agent"],
    "damaged_product": ["damaged", "broken", "cracked", "scratched", "defect"],
    "refund_return_issue": ["refund", "return", "replacement", "money back", "chargeback"],
    "app_website_issue": ["app", "website", "login", "payment error", "crash", "payment"],
}

for cat, kws in COMPLAINT_CATEGORIES.items():
    print(f"{cat:28s} → {', '.join(kws[:4])}{'...' if len(kws) > 4 else ''}")


delivery_delay               → late, delay, slow delivery, not delivered...
poor_quality                 → bad quality, broken, damaged, defective...
high_price                   → expensive, overpriced, high price, costly...
customer_service_issue       → rude, support, customer service, unhelpful...
damaged_product              → damaged, broken, cracked, scratched...
refund_return_issue          → refund, return, replacement, money back...
app_website_issue            → app, website, login, payment error...


### Step 3 — Rule-based classification

In [5]:
def classify_complaints(text: str, rules: Dict[str, List[str]] = COMPLAINT_CATEGORIES) -> List[str]:
    if not text or pd.isna(text):
        return []
    lower = str(text).lower()
    matched = [cat for cat, keywords in rules.items() if any(kw in lower for kw in keywords)]
    return matched


def assign_complaint_column(texts: pd.Series) -> pd.Series:
    def _label(t: str) -> str:
        cats = classify_complaints(t)
        return ", ".join(cats) if cats else "general"

    return texts.map(_label)


df["complaint_category"] = assign_complaint_column(df["cleaned_review_text"])

print("Complaint category counts (reviews can match multiple — counted by primary label):")
primary = df["complaint_category"].str.split(", ").str[0].value_counts().head(15)
display(primary.to_frame("count"))

display(df[["cleaned_review_text", "sentiment", "complaint_category"]].head(5))


Complaint category counts (reviews can match multiple — counted by primary label):


,count
complaint_category,
general,1384730
high_price,64003
app_website_issue,63752
delivery_delay,31466
poor_quality,12459
customer_service_issue,6764
refund_return_issue,2284
damaged_product,39


,cleaned_review_text,sentiment,complaint_category
0,switchfoot awww bummer shoulda get david carr ...,negative,general
1,upset update facebook texting might cry result...,negative,general
2,kenichan dive many time ball manage save 50 re...,negative,general
3,whole body feel itchy like fire,negative,high_price
4,nationwideclass behave mad see,negative,general


---
## Section B — TF-IDF keyword extraction

### Step 4 — TF-IDF vectorization by sentiment

Fit TF-IDF on the corpus and show the **top 20 keywords** for negative, neutral, and positive reviews.

In [6]:
def top_tfidf_keywords_by_group(
    texts: Sequence[str],
    groups: Sequence[str],
    top_n: int = TOP_N_TFIDF,
    max_features: int = 30_000,
) -> pd.DataFrame:
    rows = []
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    unique_groups = sorted(set(groups))

    for label in unique_groups:
        mask = [g == label for g in groups]
        subset = [t for t, m in zip(texts, mask) if m]
        if len(subset) < 2:
            print(f"Skipping sentiment '{label}' — not enough documents.")
            continue
        try:
            mat = vectorizer.fit_transform(subset)
        except ValueError as exc:
            print(f"TF-IDF failed for '{label}': {exc}")
            continue
        scores = np.asarray(mat.mean(axis=0)).ravel()
        terms = vectorizer.get_feature_names_out()
        top_idx = np.argsort(scores)[-top_n:][::-1]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({"sentiment": label, "rank": rank, "keyword": terms[idx], "score": round(float(scores[idx]), 6)})
    return pd.DataFrame(rows)


tfidf_by_sentiment = top_tfidf_keywords_by_group(
    df["cleaned_review_text"].tolist(),
    df["sentiment"].tolist(),
    top_n=TOP_N_TFIDF,
)

for label in tfidf_by_sentiment["sentiment"].unique():
    print(f"\nTop {TOP_N_TFIDF} keywords — {label}")
    sub = tfidf_by_sentiment[tfidf_by_sentiment["sentiment"] == label]
    display(sub[["rank", "keyword", "score"]])



Top 20 keywords — negative


,rank,keyword,score
0,1,go,0.018024
1,2,get,0.016456
2,3,miss,0.012723
3,4,work,0.012472
4,5,day,0.010384
5,6,want,0.009821
6,7,not,0.009354
7,8,like,0.009053
8,9,today,0.008677
9,10,feel,0.008469



Top 20 keywords — positive


,rank,keyword,score
20,1,thank,0.016444
21,2,go,0.014622
22,3,good,0.014617
23,4,get,0.014472
24,5,love,0.012737
25,6,day,0.011210
26,7,like,0.009012
27,8,lol,0.008901
28,9,well,0.008711
29,10,see,0.008384


### Step 5 — Keyword extraction per review

In [7]:
def extract_tfidf_keywords_per_review(
    texts: Sequence[str],
    top_n: int = TOP_N_PER_REVIEW,
    max_features: int = 20_000,
    batch_size: int = 50_000,
) -> List[str]:
    if not texts:
        return []
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    try:
        vectorizer.fit(texts)
    except ValueError as exc:
        raise RuntimeError(f"TF-IDF fit failed: {exc}") from exc

    terms = vectorizer.get_feature_names_out()
    results: List[str] = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        mat = vectorizer.transform(batch)
        for row in mat:
            arr = row.toarray().ravel()
            if arr.max() == 0:
                results.append("")
                continue
            top_idx = np.argsort(arr)[-top_n:][::-1]
            results.append(", ".join(terms[i] for i in top_idx if arr[i] > 0))
    return results


try:
    df["tfidf_keywords"] = extract_tfidf_keywords_per_review(df["cleaned_review_text"].tolist())
    print("Sample tfidf_keywords:")
    display(df[["cleaned_review_text", "tfidf_keywords"]].head(5))
except RuntimeError as exc:
    print(f"Per-review TF-IDF error: {exc}", file=sys.stderr)
    df["tfidf_keywords"] = ""
    raise


Sample tfidf_keywords:


,cleaned_review_text,tfidf_keywords
0,switchfoot awww bummer shoulda get david carr ...,"carr, shoulda, third, bummer, david"
1,upset update facebook texting might cry result...,"might cry, today also, texting, school today, ..."
2,kenichan dive many time ball manage save 50 re...,"dive, many time, manage, 50, ball"
3,whole body feel itchy like fire,"whole body, itchy, fire, body, whole"
4,nationwideclass behave mad see,"behave, mad, see"


---
## Section C — Topic modeling (LDA)

### Step 6 — Prepare text for topic modeling

Tokenize, remove stopwords, and build a document-term matrix with scikit-learn. LDA training uses `LDA_MAX_DOCS` rows for speed on large datasets; topics are then assigned to all rows.

In [9]:
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize

    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    import nltk

    nltk.download("stopwords", quiet=True)
    nltk.download("punkt", quiet=True)
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize

    STOPWORDS = set(stopwords.words("english"))

from sklearn.feature_extraction.text import CountVectorizer

TOKEN_RE = re.compile(r"^[a-z]{2,}$")


def tokenize_for_lda(text: str) -> List[str]:
    if not text or pd.isna(text):
        return []
    try:
        tokens = word_tokenize(str(text).lower())
    except Exception:
        tokens = str(text).lower().split()
    return [t for t in tokens if t not in STOPWORDS and TOKEN_RE.match(t)]


def build_lda_matrix(texts: Sequence[str]):
    tokenized = [tokenize_for_lda(t) for t in texts]
    tokenized = [t for t in tokenized if t]
    if len(tokenized) < 10:
        raise ValueError("Not enough non-empty tokenized documents for LDA.")

    vectorizer = CountVectorizer(
        analyzer=tokenize_for_lda,
        min_df=5,
        max_df=0.5,
        max_features=50_000,
    )
    matrix = vectorizer.fit_transform(texts)
    if matrix.shape[0] < 10 or matrix.shape[1] < 2:
        raise ValueError("Corpus too small after filtering for LDA.")
    return vectorizer, matrix, tokenized


lda_train_n = len(df) if LDA_MAX_DOCS is None else min(LDA_MAX_DOCS, len(df))
lda_train_df = df if lda_train_n == len(df) else df.sample(lda_train_n, random_state=RANDOM_STATE)
print(f"Preparing LDA on {len(lda_train_df):,} documents...")

lda_vectorizer, lda_train_matrix, train_tokens = build_lda_matrix(
    lda_train_df["cleaned_review_text"].tolist()
)
print(
    f"Vocabulary size: {len(lda_vectorizer.vocabulary_):,} | "
    f"training docs: {lda_train_matrix.shape[0]:,}"
)


Preparing LDA on 100,000 documents...
Vocabulary size: 9,070 | training docs: 100,000


### Step 7 — Train LDA model

In [10]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = None
coherence_score = None

try:
    lda_model = LatentDirichletAllocation(
        n_components=LDA_NUM_TOPICS,
        random_state=RANDOM_STATE,
        max_iter=LDA_PASSES,
        learning_method="batch",
        evaluate_every=-1,
    )
    lda_model.fit(lda_train_matrix)

    feature_names = lda_vectorizer.get_feature_names_out()
    print(f"\nTop words per topic ({LDA_NUM_TOPICS} topics):")
    for tid, topic in enumerate(lda_model.components_):
        top_idx = topic.argsort()[:-11:-1]
        words = ", ".join(feature_names[i] for i in top_idx)
        print(f"  Topic {tid}: {words}")

    print("\nTopic coherence (c_v): skipped (requires gensim; using sklearn LDA).")
except Exception as exc:
    print(f"LDA training failed: {exc}", file=sys.stderr)
    raise



Top words per topic (8 topics):
  Topic 0: quot, wish, could, say, like, love, lt, make, play, come
  Topic 1: get, work, go, back, home, tomorrow, hour, still, week, wait
  Topic 2: go, want, sleep, bed, really, still, feel, tired, get, talk
  Topic 3: day, well, today, happy, go, oh, time, think, feel, work
  Topic 4: good, watch, one, make, love, lol, think, find, know, look
  Topic 5: twitter, new, know, yeah, hey, would, get, love, like, yes
  Topic 6: thank, miss, amp, love, see, na, wan, go, much, hope
  Topic 7: good, night, last, morning, go, day, great, sound, like, time

Topic coherence (c_v): skipped (requires gensim; using sklearn LDA).


### Step 8 — Assign topics to reviews

In [11]:
def assign_dominant_topics(
    texts: Sequence[str],
    model: LatentDirichletAllocation,
    vectorizer: CountVectorizer,
    batch_size: int = 20_000,
) -> List[int]:
    topics: List[int] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        matrix = vectorizer.transform(batch)
        doc_topics = model.transform(matrix)
        for row_idx in range(matrix.shape[0]):
            if matrix[row_idx].nnz == 0:
                topics.append(-1)
            else:
                topics.append(int(doc_topics[row_idx].argmax()))
    return topics


if lda_model is None:
    df["lda_topic"] = -1
    print("LDA model unavailable — lda_topic set to -1.")
else:
    df["lda_topic"] = assign_dominant_topics(
        df["cleaned_review_text"].tolist(),
        lda_model,
        lda_vectorizer,
    )
    print(df["lda_topic"].value_counts().sort_index().head(15))


lda_topic
-1     14859
 0    163171
 1    211580
 2    204645
 3    212479
 4    201269
 5    225731
 6    176991
 7    154772
Name: count, dtype: int64


---
## Step 9 — Final combined output

Merge keyword complaints, TF-IDF keywords, and LDA topic assignments.

In [12]:
final_cols = ["cleaned_review_text", "complaint_category", "tfidf_keywords", "lda_topic"]
if "sentiment" in df.columns:
    final_cols.insert(1, "sentiment")
if "row_id" in df.columns:
    final_cols.insert(0, "row_id")

final_output = df[final_cols].copy()
print(f"Final shape: {final_output.shape}")
display(final_output.head(10))


Final shape: (1565497, 6)


,row_id,cleaned_review_text,sentiment,complaint_category,tfidf_keywords,lda_topic
0,1,switchfoot awww bummer shoulda get david carr ...,negative,general,"carr, shoulda, third, bummer, david",3
1,2,upset update facebook texting might cry result...,negative,general,"might cry, today also, texting, school today, ...",5
2,3,kenichan dive many time ball manage save 50 re...,negative,general,"dive, many time, manage, 50, ball",2
3,4,whole body feel itchy like fire,negative,high_price,"whole body, itchy, fire, body, whole",0
4,5,nationwideclass behave mad see,negative,general,"behave, mad, see",4
5,6,kwesidei whole crew,negative,general,"crew, whole",0
6,7,need hug,negative,general,"need hug, hug, need",6
7,8,loltrish hey long time see yes rain bit bit lo...,negative,general,"bit, fine thank, time see, long time, fine",3
8,9,tatiana k nope,negative,general,nope,5
9,10,twittera que muera,negative,general,que,7


## Step 10 — Save results

In [13]:
if OUTPUT_MAX_ROWS is not None and len(final_output) > OUTPUT_MAX_ROWS:
    final_output = final_output.head(OUTPUT_MAX_ROWS).copy()
    print(f"Capped export to {OUTPUT_MAX_ROWS:,} rows.")

try:
    final_output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"Saved {len(final_output):,} rows → {OUTPUT_CSV.name}")
except OSError as exc:
    print(f"Save failed: {exc}", file=sys.stderr)
    raise


Saved 1,565,497 rows → complaint_topic_results_phase5.csv


---
## Summary

| Section | Output |
|---------|--------|
| A | `complaint_category` (rule-based, multi-label) |
| B | TF-IDF top keywords by sentiment + `tfidf_keywords` per review |
| C | LDA topics + `lda_topic` column |
| Final | `complaint_topic_results_phase5.csv` |

**Tips:** Set `LDA_MAX_DOCS` higher for richer topics, or `MAX_ROWS` to subsample for faster experiments on the full 1.5M-row dataset.